## 欢迎进入 Notebook  

这里你可以编写代码，文档  

### 关于文件目录  


**project**：project 目录是本项目的工作空间，可以把将项目运行有关的所有文件放在这里，目录中文件的增、删、改操作都会被保留  


**input**：input 目录是数据集的挂载位置，所有挂载进项目的数据集都在这里，未挂载数据集时 input 目录被隐藏  


**temp**：temp 目录是临时磁盘空间，训练或分析过程中产生的不必要文件可以存放在这里，目录中的文件不会保存  


In [17]:
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
import re

RAW_PATH = "/home/mw/input/abc7876/副本ruc_Class25Q2_train_price_clean3.xlsx"

def cheap_xlsx_reader(xlsx_path):
    """
    极简xlsx读取器：不依赖openpyxl。
    假设：
      - 只读第一个sheet
      - 第一行是列名
      - 没有合并单元格/图片/公式引用乱七八糟
    返回 pandas.DataFrame
    """
    with zipfile.ZipFile(xlsx_path) as z:
        # 1. 读取共享字符串表 (sharedStrings.xml)，Excel会把文本存在这里
        shared_strings = []
        if "xl/sharedStrings.xml" in z.namelist():
            ss_xml = z.read("xl/sharedStrings.xml")
            ss_root = ET.fromstring(ss_xml)
            # 每个 si 下面可能有多个 t，把它们连起来
            for si in ss_root.findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}si"):
                text_parts = []
                for t in si.findall(".//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t"):
                    text_parts.append(t.text if t.text else "")
                shared_strings.append("".join(text_parts))
        else:
            shared_strings = []

        # 2. 找第一个sheet的路径，从 workbook.xml 里解析 sheetId -> sheetN.xml
        wb_xml = z.read("xl/workbook.xml")
        wb_root = ET.fromstring(wb_xml)
        ns = {"main": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

        sheets = wb_root.find("main:sheets", ns)
        first_sheet = sheets[0]
        sheet_rel_id = first_sheet.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]

        # relationships: workbook.xml.rels
        rels_xml = z.read("xl/_rels/workbook.xml.rels")
        rels_root = ET.fromstring(rels_xml)
        sheet_path = None
        for rel in rels_root.findall("{http://schemas.openxmlformats.org/package/2006/relationships}Relationship"):
            if rel.attrib["Id"] == sheet_rel_id:
                sheet_path = "xl/" + rel.attrib["Target"]
                break

        if sheet_path is None:
            raise RuntimeError("找不到sheet路径")

        # 3. 解析sheet XML
        sheet_xml = z.read(sheet_path)
        sheet_root = ET.fromstring(sheet_xml)

        # Excel单元格编码如 A1,B1,... 我们转成列索引
        def col_letter_to_index(col_letters):
            # "A"->0, "B"->1, ..., "Z"->25, "AA"->26, ...
            expn = 0
            col_idx = 0
            for char in col_letters[::-1]:
                col_idx += (ord(char.upper()) - ord("A") + 1) * (26 ** expn)
                expn += 1
            return col_idx - 1

        data_rows = []
        for row in sheet_root.findall(".//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row"):
            row_cells = {}
            for c in row.findall("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c"):
                # cell reference like "C12"
                cell_ref = c.attrib.get("r")  # e.g. "C12"
                # split letters vs numbers
                m = re.match(r"([A-Z]+)([0-9]+)", cell_ref)
                if not m:
                    continue
                col_letters = m.group(1)
                col_idx = col_letter_to_index(col_letters)

                cell_type = c.attrib.get("t")  # "s" means shared string
                v = c.find("{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v")
                cell_val = None
                if v is not None and v.text is not None:
                    if cell_type == "s":
                        # shared string lookup
                        ss_idx = int(v.text)
                        if ss_idx < len(shared_strings):
                            cell_val = shared_strings[ss_idx]
                        else:
                            cell_val = ""
                    else:
                        # number or inline
                        cell_val = v.text
                else:
                    cell_val = ""

                row_cells[col_idx] = cell_val
            # convert row_cells dict -> list, filling gaps
            if row_cells:
                max_col = max(row_cells.keys())
                row_list = [row_cells.get(ci, "") for ci in range(max_col+1)]
                data_rows.append(row_list)

        # 第一行=列名
        header = data_rows[0]
        body   = data_rows[1:]

        df_out = pd.DataFrame(body, columns=header)
        return df_out

df = cheap_xlsx_reader(RAW_PATH)

print("列名预览：")
print(df.columns.tolist()[:50])
print(df.head())

列名预览：
['区域', '板块', 'Price', '建筑面积', '交易时间', 'lon', 'lat', '房屋总数', 'subway', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '室', '厅', '厨', '卫', '地下室_01', '底层_01', '低楼层_01', '高楼层_01', '顶层_01', '配备电梯_无_01', '朝向_东_01', '朝向_南_01', '朝向_西_01', '朝向_北_01', '建筑结构_混合结构_01', '建筑结构_未知结构_01', '建筑结构_框架结构_01', '建筑结构_钢结构_01', '建筑结构_砖混结构_01', '装修_简装_01', '装修_毛坯_01', '装修_其他_01', '房龄', 'city_00', 'city_01', 'city_03', 'city_04', 'city_05', 'city_06', 'city_07', 'city_08', 'city_09', 'city_10', 'city_11']
    区域    板块        Price    建筑面积   交易时间          lon          lat  房屋总数  \
0  109   150  6194048.992    52.3  44256  117.4242785  40.97575181  1317   
1   65   299  4354153.263  127.44  44105  117.3892279   41.0912954  2317   
2   62   911  3321991.616  118.02  44136  117.2009335  40.74791948  1554   
3  123  1102  7895655.584  293.23  44986  117.7673081  41.22880344    66   
4   81   295  1902960.295   39.85  43800  117.3345301  40.95252997  1685   

  subway 楼栋总数  ... city_01 city_03 city_04 city_05 city_06 city_0

In [20]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 0. 数值列转 numeric，先清洗
########################################
num_like_cols = [
    "Price","建筑面积","lon","lat",
    "房屋总数","subway","楼栋总数","绿 化 率","容 积 率","物 业 费",
    "室","厅","厨","卫",
    "地下室_01","底层_01","低楼层_01","高楼层_01","顶层_01","配备电梯_无_01",
    "朝向_东_01","朝向_南_01","朝向_西_01","朝向_北_01",
    "装修_简装_01","装修_毛坯_01","装修_其他_01",
    "房龄",
    "time_index",
    "city_00","city_01","city_03","city_04","city_05","city_06",
    "city_07","city_08","city_09","city_10","city_11"
]
num_like_cols = [c for c in num_like_cols if c in df.columns]

for c in num_like_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

########################################
# 1. 基于板块形成 block_profile
########################################
if "配备电梯_无_01" in df.columns:
    df["有电梯_01"] = 1 - df["配备电梯_无_01"]
else:
    # 如果本来就没有这个列，兜个底，设成0
    df["有电梯_01"] = 0

if "板块" not in df.columns:
    raise ValueError("你的 df 里没有 '板块' 这一列，我没法按板块聚类。需要检查列名是不是改了，比如叫'区域'之类的。")

block_profile = (
    df.groupby("板块")
    .agg(
        n_obs            = ("板块","size"),
        center_lon       = ("lon","mean"),
        center_lat       = ("lat","mean"),
        avg_price        = ("Price","mean"),
        avg_age          = ("房龄","mean"),
        avg_green        = ("绿 化 率","mean"),
        avg_plot_ratio   = ("容 积 率","mean"),
        avg_property_fee = ("物 业 费","mean"),
        avg_subway       = ("subway","mean"),
        share_elevator   = ("有电梯_01","mean"),
    )
    .reset_index()
)

# 清洗 block_profile 的数值列
for c in [
    "center_lon","center_lat","avg_price","avg_age",
    "avg_green","avg_plot_ratio","avg_property_fee",
    "avg_subway","share_elevator"
]:
    block_profile[c] = pd.to_numeric(block_profile[c], errors="coerce")
    block_profile[c] = block_profile[c].fillna(block_profile[c].median())

########################################
# 2. KMeans 聚类 (K=30，按板块聚)
########################################
cluster_features = [
    "center_lon","center_lat",
    "avg_price","avg_age",
    "avg_green","avg_plot_ratio",
    "avg_property_fee","avg_subway","share_elevator"
]

X_block = block_profile[cluster_features].values
scaler_block = StandardScaler()
X_block_scaled = scaler_block.fit_transform(X_block)

K = min(30, len(block_profile))
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
block_profile["market_cluster30"] = kmeans.fit_predict(X_block_scaled).astype(int)

########################################
# 3. 把聚类结果合回 df
########################################
df = df.merge(
    block_profile[["板块","market_cluster30"]],
    on="板块",
    how="left"
)

# 安全检查：有些情况下 merge 失败会导致全是 NaN
if "market_cluster30" not in df.columns or df["market_cluster30"].isna().all():
    # fallback：如果还没有这个列，或者全是NaN
    # 我们用KMeans直接在行级别（经纬度+房价等）聚一下，至少保证代码后面能跑
    print("警告: 按板块聚类的 market_cluster30 没并进来，我用房源级别的临时聚类代替。")

    fallback_feats = []
    for cc in ["lon","lat","Price","房龄","绿 化 率","容 积 率","物 业 费","subway","有电梯_01"]:
        if cc in df.columns:
            fallback_feats.append(cc)

    if len(fallback_feats) == 0:
        raise ValueError("连 fallback 聚类的基础特征都没有，没法继续。")

    tmp_X = df[fallback_feats].copy()
    for c in tmp_X.columns:
        tmp_X[c] = pd.to_numeric(tmp_X[c], errors="coerce")
    tmp_X = tmp_X.fillna(tmp_X.median(numeric_only=True))

    scaler_row = StandardScaler()
    tmp_X_scaled = scaler_row.fit_transform(tmp_X)

    K_fallback = min(30, len(df))
    kmeans_row = KMeans(n_clusters=K_fallback, random_state=42, n_init=10)
    df["market_cluster30"] = kmeans_row.fit_predict(tmp_X_scaled).astype(int)

########################################
# 4. cluster30 做成哑变量
########################################
cluster_dummies = pd.get_dummies(
    df["market_cluster30"],
    prefix="cluster30",
    drop_first=False
).astype(int)

########################################
# 5. 构建特征矩阵 X_full 和 y_all
########################################
core_cols = [
    "建筑面积","房屋总数","楼栋总数","绿 化 率","容 积 率","物 业 费",
    "subway",
    "室","厅","厨","卫",
    "地下室_01","底层_01","低楼层_01","高楼层_01","顶层_01",
    "配备电梯_无_01",
    "朝向_东_01","朝向_南_01","朝向_西_01","朝向_北_01",
    "装修_简装_01","装修_毛坯_01","装修_其他_01",
    "房龄",
    "time_index",
    "city_00","city_01","city_03","city_04","city_05","city_06",
    "city_07","city_08","city_09","city_10","city_11"
]
core_cols = [c for c in core_cols if c in df.columns]

X_core = df[core_cols].copy()
for c in X_core.columns:
    X_core[c] = pd.to_numeric(X_core[c], errors="coerce")
X_core = X_core.fillna(X_core.median(numeric_only=True))

X_full = pd.concat(
    [X_core.reset_index(drop=True),
     cluster_dummies.reset_index(drop=True)],
    axis=1
)

# 丢掉全是常数的列
X_full = X_full.loc[:, X_full.std(axis=0) > 0]

y_all = pd.to_numeric(df["Price"], errors="coerce")
y_all = y_all.fillna(y_all.median())

########################################
# 6. IQR 过滤异常值
########################################
Q1 = y_all.quantile(0.25)
Q3 = y_all.quantile(0.75)
IQR = Q3 - Q1
lower_cut = Q1 - 1.5 * IQR
upper_cut = Q3 + 1.5 * IQR
mask_ok = (y_all >= lower_cut) & (y_all <= upper_cut)

X_ok = X_full.loc[mask_ok].reset_index(drop=True)
y_ok = y_all.loc[mask_ok].reset_index(drop=True)

print("用于最终线性模型的样本量:", X_ok.shape[0], "特征数:", X_ok.shape[1])

########################################
# 7. train / test 划分
########################################
X_train, X_test, y_train, y_test = train_test_split(
    X_ok, y_ok,
    test_size=0.2,
    random_state=111
)

########################################
# 8. OLS (线性回归) 管道
########################################
ols_pipe = make_pipeline(
    # with_mean=False 对稀疏/哑变量友好
    StandardScaler(with_mean=False),
    LinearRegression()
)
ols_pipe.fit(X_train, y_train)

########################################
# 9. 训练集 / 测试集指标
########################################
y_pred_tr = ols_pipe.predict(X_train)
train_MAE  = mean_absolute_error(y_train, y_pred_tr)
train_MSE  = mean_squared_error(y_train, y_pred_tr)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_tr)

y_pred_te = ols_pipe.predict(X_test)
test_MAE  = mean_absolute_error(y_test, y_pred_te)
test_MSE  = mean_squared_error(y_test, y_pred_te)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_te)

print("====== OLS (K=30簇 + 结构 + 时间趋势 + 城市固定效应) ======")
print("[训练集 / in-sample]")
print(f"MAE  : {train_MAE:.4f}")
print(f"MSE  : {train_MSE:.4f}")
print(f"RMSE : {train_RMSE:.4f}")
print(f"R^2  : {train_R2:.4f}")
print("")
print("[测试集 / out-of-sample]")
print(f"MAE  : {test_MAE:.4f}")
print(f"MSE  : {test_MSE:.4f}")
print(f"RMSE : {test_RMSE:.4f}")
print(f"R^2  : {test_R2:.4f}")
print("=======================================================")

########################################
# 10. 6折交叉验证 (计算 MAE / MSE / RMSE / R²)
########################################
kf = KFold(n_splits=6, shuffle=True, random_state=111)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1
    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_pipeline(
        StandardScaler(with_mean=False),
        LinearRegression()
    )
    pipe_cv.fit(X_tr, y_tr)
    y_hat = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f"[CV折 {fold_id}/6] "
          f"MAE={mae_val:.4f} | MSE={mse_val:.4f} | RMSE={rmse_val:.4f} | R^2={r2_val:.4f}")

cv_MAE_mean   = float(np.mean(cv_mae_list))
cv_MAE_std    = float(np.std(cv_mae_list))
cv_MSE_mean   = float(np.mean(cv_mse_list))
cv_MSE_std    = float(np.std(cv_mse_list))
cv_RMSE_mean  = float(np.mean(cv_rmse_list))
cv_RMSE_std   = float(np.std(cv_rmse_list))
cv_R2_mean    = float(np.mean(cv_r2_list))
cv_R2_std     = float(np.std(cv_r2_list))

print("\n[6折交叉验证 - 汇总平均±std]")
print(f"CV MAE   : {cv_MAE_mean:.4f} (std {cv_MAE_std:.4f})")
print(f"CV MSE   : {cv_MSE_mean:.4f} (std {cv_MSE_std:.4f})")
print(f"CV RMSE  : {cv_RMSE_mean:.4f} (std {cv_RMSE_std:.4f})")
print(f"CV R^2   : {cv_R2_mean:.4f} (std {cv_R2_std:.4f})")
print("=======================================================")

########################################
# 11. 导出结果
########################################
df_clean = df.loc[mask_ok].reset_index(drop=True)
cluster_dummies_clean = cluster_dummies.loc[mask_ok].reset_index(drop=True)

pred_all = ols_pipe.predict(X_ok)

export_df = pd.concat(
    [
        df_clean.reset_index(drop=True),
        cluster_dummies_clean.reset_index(drop=True),
        pd.Series(pred_all, name="OLS_predicted_price_after_cluster30")
    ],
    axis=1
)

export_df.to_csv("housing_with_cluster30.csv", index=False)
print("导出完成 -> housing_with_cluster30.csv")


用于最终线性模型的样本量: 96048 特征数: 66
====== OLS (K=30簇 + 结构 + 时间趋势 + 城市固定效应) ======
[训练集 / in-sample]
MAE  : 485454.7869
MSE  : 466389246958.5073
RMSE : 682926.9704
R^2  : 0.6473

[测试集 / out-of-sample]
MAE  : 484273.2277
MSE  : 460572713480.1712
RMSE : 678655.0770
R^2  : 0.6494
[CV折 1/6] MAE=482767.0730 | MSE=455043904581.8588 | RMSE=674569.4216 | R^2=0.6525
[CV折 2/6] MAE=487954.2458 | MSE=472225505500.0215 | RMSE=687186.6599 | R^2=0.6434
[CV折 3/6] MAE=485318.1136 | MSE=466532855851.8430 | RMSE=683032.1046 | R^2=0.6457
[CV折 4/6] MAE=486454.0736 | MSE=470033220881.3555 | RMSE=685589.6884 | R^2=0.6489
[CV折 5/6] MAE=487710.9302 | MSE=469909850354.9074 | RMSE=685499.7085 | R^2=0.6456
[CV折 6/6] MAE=483459.1005 | MSE=461725753582.9575 | RMSE=679504.0497 | R^2=0.6474

[6折交叉验证 - 汇总平均±std]
CV MAE   : 485610.5894 (std 1975.5022)
CV MSE   : 465911848458.8240 (std 5899904513.9866)
CV RMSE  : 682563.6054 (std 4332.7793)
CV R^2   : 0.6472 (std 0.0029)
导出完成 -> housing_with_cluster30.csv


In [21]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import KFold
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

########################################################
# 可调参数
########################################################
ALPHAS = [1.0, 5.0, 10.0,100.0, 1000.0, 10000.0]   # 想加就加，比如 50.0,100.0
SUBSET_SIZE = 2000          # 只用这么多行来调参，防止卡
KF_SPLITS = 6               # 6 折交叉验证
SEED = 42

########################################################
# 第0步：基本信息
########################################################
print("第0步：开始小样本调参流程(只挑alpha，不训练全量)", flush=True)

########################################################
# 第1步：检查 X_ok / y_ok
########################################################
print("第1步：检查 X_ok / y_ok 类型...", flush=True)

print("  原始类型:", flush=True)
print("   type(X_ok) =", type(X_ok), flush=True)
print("   type(y_ok) =", type(y_ok), flush=True)

# 如果不是DataFrame/Series，转一下，保证后面 iloc 可用
if not isinstance(X_ok, pd.DataFrame):
    print("  X_ok 不是DataFrame，自动转成DataFrame", flush=True)
    X_ok = pd.DataFrame(X_ok)

if not isinstance(y_ok, pd.Series):
    print("  y_ok 不是Series，自动转成Series", flush=True)
    y_ok = pd.Series(y_ok)

print("  转换后:", flush=True)
print("   type(X_ok) =", type(X_ok), flush=True)
print("   type(y_ok) =", type(y_ok), flush=True)
print("   X_ok.shape =", X_ok.shape, flush=True)
print("   y_ok.shape =", y_ok.shape, flush=True)

########################################################
# 第2步：抽子样本（比如2000行）避免整机爆掉
########################################################
print("第2步：随机抽子样本...", flush=True)

n_total = len(X_ok)
n_take  = min(SUBSET_SIZE, n_total)

rng = np.random.RandomState(SEED)
idx = rng.choice(n_total, size=n_take, replace=False)

X_sub = X_ok.iloc[idx].copy()
y_sub = y_ok.iloc[idx].copy()

print(f"  子样本大小 = {len(X_sub)} / 全部 {n_total} 行", flush=True)

########################################################
# 第3步：建立 6 折划分 和 标准化器
########################################################
print("第3步：准备交叉验证(6折)和标准化...", flush=True)

kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=111)
scaler = StandardScaler()

print("  将会尝试的 alpha 列表 =", ALPHAS, flush=True)

########################################################
# 第4步：对每个 alpha 做交叉验证
########################################################
print("第4步：开始扫描 alpha ...", flush=True)

alpha_records = []

for a in ALPHAS:
    print(f"\n=== Alpha = {a} 开始 ===", flush=True)
    t_alpha = time.time()

    fold_mae_list = []
    fold_r2_list  = []

    fold_id = 0
    for tr_idx, va_idx in kf.split(X_sub):
        fold_id += 1
        t_fold = time.time()

        X_tr = X_sub.iloc[tr_idx]
        X_va = X_sub.iloc[va_idx]
        y_tr = y_sub.iloc[tr_idx]
        y_va = y_sub.iloc[va_idx]

        # 管道：先标准化，再Lasso
        pipe = Pipeline([
            ("scaler", scaler),
            ("lasso",  Lasso(
                alpha=a,
                fit_intercept=True,
                max_iter=50000,
                random_state=42
            ))
        ])

        pipe.fit(X_tr, y_tr)
        y_hat = pipe.predict(X_va)

        mae_val = mean_absolute_error(y_va, y_hat)
        r2_val  = r2_score(y_va, y_hat)

        fold_mae_list.append(mae_val)
        fold_r2_list.append(r2_val)

        print(f"   折 {fold_id}/{KF_SPLITS} | MAE={mae_val:,.4f} | R²={r2_val:.4f} | 本折耗时 {time.time()-t_fold:.2f}s", flush=True)

    mae_mean = float(np.mean(fold_mae_list))
    mae_std  = float(np.std(fold_mae_list))
    r2_mean  = float(np.mean(fold_r2_list))
    r2_std   = float(np.std(fold_r2_list))
    elapsed_alpha = time.time() - t_alpha

    print(f"=== Alpha = {a} 结束 ===", flush=True)
    print(f"    -> CV 平均MAE={mae_mean:,.4f} ± {mae_std:,.4f}", flush=True)
    print(f"    -> CV 平均R² ={r2_mean:.4f} ± {r2_std:.4f}", flush=True)
    print(f"    -> 该alpha总耗时 {elapsed_alpha:.2f}s", flush=True)

    alpha_records.append({
        "alpha": a,
        "cv_mae_mean": mae_mean,
        "cv_mae_std":  mae_std,
        "cv_r2_mean":  r2_mean,
        "cv_r2_std":   r2_std,
        "time_sec": elapsed_alpha
    })

########################################################
# 第5步：总结结果，选最好的 alpha
########################################################
print("\n第5步：总结所有 alpha 的表现并选最优...", flush=True)

results_df = pd.DataFrame(alpha_records)
results_sorted = results_df.sort_values("cv_mae_mean").reset_index(drop=True)

best_row = results_sorted.iloc[0]
best_alpha = best_row["alpha"]

print("========== 小样本调参结果汇总(按MAE从小到大排序) ==========", flush=True)
print(results_sorted.to_string(index=False), flush=True)
print("=========================================================", flush=True)
print(f"\n>>> 结论：小样本里最好的 alpha = {best_alpha}", flush=True)

print("\n第5步完成。后续你就把这个 best_alpha 拿去全量训练/算train&test指标就可以了。", flush=True)


第0步：开始小样本调参流程(只挑alpha，不训练全量)
第1步：检查 X_ok / y_ok 类型...
  原始类型:
   type(X_ok) = <class 'pandas.core.frame.DataFrame'>
   type(y_ok) = <class 'pandas.core.series.Series'>
  转换后:
   type(X_ok) = <class 'pandas.core.frame.DataFrame'>
   type(y_ok) = <class 'pandas.core.series.Series'>
   X_ok.shape = (96048, 66)
   y_ok.shape = (96048,)
第2步：随机抽子样本...
  子样本大小 = 2000 / 全部 96048 行
第3步：准备交叉验证(6折)和标准化...
  将会尝试的 alpha 列表 = [1.0, 5.0, 10.0, 100.0, 1000.0, 10000.0]
第4步：开始扫描 alpha ...

=== Alpha = 1.0 开始 ===
   折 1/6 | MAE=521,550.1515 | R²=0.5956 | 本折耗时 20.16s
   折 2/6 | MAE=502,726.7401 | R²=0.6340 | 本折耗时 37.41s


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.853e+11, tolerance: 2.298e+11
  model = cd_fast.enet_coordinate_descent(


   折 3/6 | MAE=539,458.1930 | R²=0.5977 | 本折耗时 37.70s


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.408e+13, tolerance: 2.288e+11
  model = cd_fast.enet_coordinate_descent(


   折 4/6 | MAE=487,807.7385 | R²=0.6577 | 本折耗时 37.81s


...

   折 4/6 | MAE=487,933.8427 | R²=0.6577 | 本折耗时 20.01s
   折 5/6 | MAE=504,608.8777 | R²=0.6566 | 本折耗时 8.71s
   折 6/6 | MAE=476,143.4125 | R²=0.6873 | 本折耗时 39.42s
=== Alpha = 5.0 结束 ===
    -> CV 平均MAE=505,392.4132 ± 20,763.0888
    -> CV 平均R² =0.6382 ± 0.0331
    -> 该alpha总耗时 149.30s

=== Alpha = 10.0 开始 ===


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.504e+12, tolerance: 2.308e+11
  model = cd_fast.enet_coordinate_descent(


   折 1/6 | MAE=521,505.6124 | R²=0.5957 | 本折耗时 4.00s
   折 2/6 | MAE=502,699.2955 | R²=0.6341 | 本折耗时 35.90s


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:634: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.880e+11, tolerance: 2.298e+11
  model = cd_fast.enet_coordinate_descent(


   折 3/6 | MAE=539,383.3389 | R²=0.5979 | 本折耗时 19.41s
   折 4/6 | MAE=487,962.8332 | R²=0.6576 | 本折耗时 9.72s
   折 5/6 | MAE=504,615.8697 | R²=0.6566 | 本折耗时 6.01s
   折 6/6 | MAE=476,130.1014 | R²=0.6873 | 本折耗时 20.30s
=== Alpha = 10.0 结束 ===
    -> CV 平均MAE=505,382.8419 ± 20,748.2799
    -> CV 平均R² =0.6382 ± 0.0331
    -> 该alpha总耗时 95.61s

=== Alpha = 100.0 开始 ===
   折 1/6 | MAE=521,324.1266 | R²=0.5959 | 本折耗时 2.10s
   折 2/6 | MAE=502,424.8347 | R²=0.6347 | 本折耗时 4.70s
   折 3/6 | MAE=538,699.4249 | R²=0.5994 | 本折耗时 2.10s
   折 4/6 | MAE=488,470.1954 | R²=0.6574 | 本折耗时 1.10s
   折 5/6 | MAE=504,681.2812 | R²=0.6570 | 本折耗时 1.19s
   折 6/6 | MAE=475,934.8641 | R²=0.6875 | 本折耗时 1.41s
=== Alpha = 100.0 结束 ===
    -> CV 平均MAE=505,255.7878 ± 20,520.2559
    -> CV 平均R² =0.6387 ± 0.0328
    -> 该alpha总耗时 12.70s

=== Alpha = 1000.0 开始 ===
   折 1/6 | MAE=520,263.8446 | R²=0.5962 | 本折耗时 0.25s
   折 2/6 | MAE=500,392.7286 | R²=0.6385 | 本折耗时 0.41s
   折 3/6 | MAE=536,694.8092 | R²=0.6030 | 本折耗时 0.51s
   折 4/6 

In [22]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################
# 配置
########################################
BEST_ALPHA    = 10000.0      # 你调参得出来的最优 alpha
TEST_SIZE     = 0.2          # 训练 / 测试 = 80% / 20%
KF_SPLITS     = 6            # 6 折交叉验证
RANDOM_STATE  = 111          # 保持复现

print("第0步：准备全量评估，alpha=", BEST_ALPHA, flush=True)

########################################
# 1. 确保 X_ok / y_ok 是 DataFrame / Series
########################################
print("第1步：检查 X_ok / y_ok ...", flush=True)

if not isinstance(X_ok, pd.DataFrame):
    print("  X_ok 不是 DataFrame，自动转成 DataFrame", flush=True)
    X_ok = pd.DataFrame(X_ok)

if not isinstance(y_ok, pd.Series):
    print("  y_ok 不是 Series，自动转成 Series", flush=True)
    y_ok = pd.Series(y_ok)

print("  X_ok.shape =", X_ok.shape, flush=True)
print("  y_ok.shape =", y_ok.shape, flush=True)

########################################
# 2. train / test 切分
########################################
print("第2步：切分训练集/测试集...", flush=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_ok,
    y_ok,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("  训练集大小:", X_train.shape, y_train.shape, flush=True)
print("  测试集大小:", X_test.shape,  y_test.shape,  flush=True)

########################################
# 3. 定义 Lasso 管道
########################################
def make_pipe(alpha):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lasso",  Lasso(
            alpha=alpha,
            fit_intercept=True,
            max_iter=50000,
            random_state=42
        ))
    ])

final_pipe = make_pipe(BEST_ALPHA)

########################################
# 4. 拟合训练集 (样本内模型)
########################################
print("第3步：拟合训练集...", flush=True)
t0 = time.time()
final_pipe.fit(X_train, y_train)
print(f"  拟合完成，用时 {time.time()-t0:.2f} 秒", flush=True)

########################################
# 5. 训练集 / 测试集 指标
########################################
print("第4步：计算训练集/测试集指标...", flush=True)

# 训练集预测
y_pred_train = final_pipe.predict(X_train)

train_MAE  = mean_absolute_error(y_train, y_pred_train)
train_MSE  = mean_squared_error(y_train, y_pred_train)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_train)

# 测试集预测
y_pred_test = final_pipe.predict(X_test)

test_MAE  = mean_absolute_error(y_test, y_pred_test)
test_MSE  = mean_squared_error(y_test, y_pred_test)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_test)

print("  【训练集 / in-sample】", flush=True)
print(f"     MAE  : {train_MAE:,.4f}", flush=True)
print(f"     MSE  : {train_MSE:,.4f}", flush=True)
print(f"     RMSE : {train_RMSE:,.4f}", flush=True)
print(f"     R^2  : {train_R2:.4f}", flush=True)

print("  【测试集 / out-of-sample】", flush=True)
print(f"     MAE  : {test_MAE:,.4f}", flush=True)
print(f"     MSE  : {test_MSE:,.4f}", flush=True)
print(f"     RMSE : {test_RMSE:,.4f}", flush=True)
print(f"     R^2  : {test_R2:.4f}", flush=True)

########################################
# 6. 全量数据上做 6 折交叉验证（固定 alpha，不再调参）
########################################
print("第5步：全量 6 折交叉验证...", flush=True)

kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1
    t_fold = time.time()

    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_pipe(BEST_ALPHA)
    pipe_cv.fit(X_tr, y_tr)
    y_hat = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f"    折 {fold_id}/{KF_SPLITS} | "
          f"MAE={mae_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
          f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

# 汇总6折
cv_MAE_mean  = float(np.mean(cv_mae_list))
cv_MAE_std   = float(np.std(cv_mae_list))
cv_MSE_mean  = float(np.mean(cv_mse_list))
cv_MSE_std   = float(np.std(cv_mse_list))
cv_RMSE_mean = float(np.mean(cv_rmse_list))
cv_RMSE_std  = float(np.std(cv_rmse_list))
cv_R2_mean   = float(np.mean(cv_r2_list))
cv_R2_std    = float(np.std(cv_r2_list))

print("\n第6步：6折交叉验证平均结果", flush=True)
print(f"    CV MAE   : {cv_MAE_mean:,.4f}  (std {cv_MAE_std:,.4f})", flush=True)
print(f"    CV MSE   : {cv_MSE_mean:,.4f}  (std {cv_MSE_std:,.4f})", flush=True)
print(f"    CV RMSE  : {cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f})", flush=True)
print(f"    CV R^2   : {cv_R2_mean:.4f}    (std {cv_R2_std:.4f})", flush=True)

########################################
# 7. 系数（重要特征）
########################################
lasso_model = final_pipe.named_steps["lasso"]

coef_table = pd.DataFrame({
    "feature": X_ok.columns,
    "coef": lasso_model.coef_
}).sort_values(
    by="coef",
    key=lambda s: s.abs(),
    ascending=False
)

print("\n第7步：Lasso 系数Top30（按绝对值排序）", flush=True)
print(coef_table.head(30).to_string(index=False), flush=True)

coef_table.to_csv("lasso_coef_alpha10000_full.csv", index=False)
print("系数表已保存 -> lasso_coef_alpha10000_full.csv", flush=True)

########################################
# 8. 最终总结（直接抄进报告）
########################################
print("\n第8步：最终总结（作业表抄这里）", flush=True)

print("【训练集 / in-sample】", flush=True)
print(f" MAE={train_MAE:,.4f} | MSE={train_MSE:,.4f} | RMSE={train_RMSE:,.4f} | R²={train_R2:.4f}", flush=True)

print("【测试集 / out-of-sample】", flush=True)
print(f" MAE={test_MAE:,.4f} | MSE={test_MSE:,.4f} | RMSE={test_RMSE:,.4f} | R²={test_R2:.4f}", flush=True)

print("【6折交叉验证 / 全量X_ok,y_ok】", flush=True)
print(f" MAE均值={cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f}) | "
      f"MSE均值={cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f}) | "
      f"RMSE均值={cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f}) | "
      f"R²均值={cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

print("\n✅ 全流程完成。", flush=True)


第0步：准备全量评估，alpha= 10000.0
第1步：检查 X_ok / y_ok ...
  X_ok.shape = (96048, 66)
  y_ok.shape = (96048,)
第2步：切分训练集/测试集...
  训练集大小: (76838, 66) (76838,)
  测试集大小: (19210, 66) (19210,)
第3步：拟合训练集...
  拟合完成，用时 9.54 秒
第4步：计算训练集/测试集指标...
  【训练集 / in-sample】
     MAE  : 485,891.1091
     MSE  : 474,390,664,838.7816
     RMSE : 688,760.2376
     R^2  : 0.6413
  【测试集 / out-of-sample】
     MAE  : 484,906.2912
     MSE  : 470,105,994,378.8962
     RMSE : 685,642.7600
     R^2  : 0.6422
第5步：全量 6 折交叉验证...
    折 1/6 | MAE=482,890.8502 | RMSE=681,164.7915 | R²=0.6457 | 用时 9.90s
    折 2/6 | MAE=487,510.0685 | RMSE=692,013.3912 | R²=0.6384 | 用时 15.21s
    折 3/6 | MAE=486,131.7479 | RMSE=689,408.5548 | R²=0.6390 | 用时 9.20s
    折 4/6 | MAE=487,388.8758 | RMSE=691,835.6395 | R²=0.6424 | 用时 10.41s
    折 5/6 | MAE=487,672.5556 | RMSE=691,645.8272 | R²=0.6392 | 用时 12.20s
    折 6/6 | MAE=483,469.0725 | RMSE=684,583.3697 | R²=0.6421 | 用时 12.71s

第6步：6折交叉验证平均结果
    CV MAE   : 485,843.8617  (std 1,955.7698)
    CV MSE

In [23]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################################
# 可调参数（Ridge的alpha就是L2正则强度）
########################################################
RIDGE_ALPHAS = [1.0, 10.0, 100.0, 1000.0, 10000.0]
SUBSET_SIZE  = 2000     # 跟Lasso一样，用子样本挑参，防止全量直接卡死
KF_SPLITS    = 6
SEED         = 42

print("【Ridge 第0步】开始用小样本调参 (只挑 alpha，不训练全量)", flush=True)

########################################################
# 第1步：检查 / 规整 X_ok, y_ok
########################################################
print("【Ridge 第1步】检查 X_ok / y_ok", flush=True)

if not isinstance(X_ok, pd.DataFrame):
    print("  X_ok 不是 DataFrame，自动转一下", flush=True)
    X_ok = pd.DataFrame(X_ok)

if not isinstance(y_ok, pd.Series):
    print("  y_ok 不是 Series，自动转一下", flush=True)
    y_ok = pd.Series(y_ok)

print("  X_ok.shape =", X_ok.shape, flush=True)
print("  y_ok.shape =", y_ok.shape, flush=True)

########################################################
# 第2步：抽子样本
########################################################
print("【Ridge 第2步】抽子样本做交叉验证...", flush=True)

n_total = len(X_ok)
n_take  = min(SUBSET_SIZE, n_total)

rng = np.random.RandomState(SEED)
idx = rng.choice(n_total, size=n_take, replace=False)

X_sub = X_ok.iloc[idx].copy()
y_sub = y_ok.iloc[idx].copy()

print(f"  抽到子样本: {len(X_sub)} / {n_total} 行", flush=True)

########################################################
# 第3步：6折划分 + 标准化
########################################################
kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=111)
scaler = StandardScaler()

print("  将会尝试的 Ridge alpha 列表:", RIDGE_ALPHAS, flush=True)

########################################################
# 第4步：对每个 alpha 做 6 折交叉验证
########################################################
alpha_records_ridge = []
print("【Ridge 第3步】开始扫 alpha ...", flush=True)

for a in RIDGE_ALPHAS:
    print(f"\n=== Ridge alpha = {a} 开始 ===", flush=True)
    t_alpha = time.time()

    fold_mae_list = []
    fold_mse_list = []
    fold_rmse_list = []
    fold_r2_list  = []

    fold_id = 0
    for tr_idx, va_idx in kf.split(X_sub):
        fold_id += 1
        t_fold = time.time()

        X_tr = X_sub.iloc[tr_idx]
        X_va = X_sub.iloc[va_idx]
        y_tr = y_sub.iloc[tr_idx]
        y_va = y_sub.iloc[va_idx]

        pipe = Pipeline([
            ("scaler", scaler),
            ("ridge", Ridge(
                alpha=a,
                fit_intercept=True,
                random_state=42
            ))
        ])

        pipe.fit(X_tr, y_tr)
        y_hat = pipe.predict(X_va)

        mae_val  = mean_absolute_error(y_va, y_hat)
        mse_val  = mean_squared_error(y_va, y_hat)
        rmse_val = np.sqrt(mse_val)
        r2_val   = r2_score(y_va, y_hat)

        fold_mae_list.append(mae_val)
        fold_mse_list.append(mse_val)
        fold_rmse_list.append(rmse_val)
        fold_r2_list.append(r2_val)

        print(f"   折 {fold_id}/{KF_SPLITS} | "
              f"MAE={mae_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
              f"| 本折耗时 {time.time()-t_fold:.2f}s", flush=True)

    mae_mean   = float(np.mean(fold_mae_list))
    mae_std    = float(np.std(fold_mae_list))
    mse_mean   = float(np.mean(fold_mse_list))
    mse_std    = float(np.std(fold_mse_list))
    rmse_mean  = float(np.mean(fold_rmse_list))
    rmse_std   = float(np.std(fold_rmse_list))
    r2_mean    = float(np.mean(fold_r2_list))
    r2_std     = float(np.std(fold_r2_list))
    elapsed    = time.time() - t_alpha

    print(f"=== Ridge alpha = {a} 完成 ===", flush=True)
    print(f"    -> CV 平均MAE={mae_mean:,.4f} ± {mae_std:,.4f}", flush=True)
    print(f"    -> CV 平均MSE={mse_mean:,.4f} ± {mse_std:,.4f}", flush=True)
    print(f"    -> CV 平均RMSE={rmse_mean:,.4f} ± {rmse_std:,.4f}", flush=True)
    print(f"    -> CV 平均R²={r2_mean:.4f} ± {r2_std:.4f}", flush=True)
    print(f"    -> alpha总耗时 {elapsed:.2f}s", flush=True)

    alpha_records_ridge.append({
        "alpha": a,
        "cv_mae_mean": mae_mean,
        "cv_mae_std":  mae_std,
        "cv_mse_mean": mse_mean,
        "cv_mse_std":  mse_std,
        "cv_rmse_mean": rmse_mean,
        "cv_rmse_std":  rmse_std,
        "cv_r2_mean":  r2_mean,
        "cv_r2_std":   r2_std,
        "time_sec": elapsed
    })

########################################################
# 第5步：输出排序表，选最优 alpha
########################################################
print("\n【Ridge 第4步】按 MAE 从小到大排序，选最好的 alpha", flush=True)

ridge_results_df = pd.DataFrame(alpha_records_ridge)
ridge_sorted = ridge_results_df.sort_values("cv_mae_mean").reset_index(drop=True)

best_ridge_row = ridge_sorted.iloc[0]
best_ridge_alpha = best_ridge_row["alpha"]

print("========== Ridge 小样本调参结果 (MAE升序) ==========", flush=True)
print(ridge_sorted.to_string(index=False), flush=True)
print("===================================================", flush=True)

print(f"\n>>> Ridge 最优 alpha(基于子样本CV) = {best_ridge_alpha}", flush=True)

# 你会用这个 best_ridge_alpha 去做后面的“全量评估”


【Ridge 第0步】开始用小样本调参 (只挑 alpha，不训练全量)
【Ridge 第1步】检查 X_ok / y_ok
  X_ok.shape = (96048, 66)
  y_ok.shape = (96048,)
【Ridge 第2步】抽子样本做交叉验证...
  抽到子样本: 2000 / 96048 行
  将会尝试的 Ridge alpha 列表: [1.0, 10.0, 100.0, 1000.0, 10000.0]
【Ridge 第3步】开始扫 alpha ...

=== Ridge alpha = 1.0 开始 ===
   折 1/6 | MAE=521,953.5545 | RMSE=722,053.6153 | R²=0.5942 | 本折耗时 0.02s
   折 2/6 | MAE=503,072.5831 | RMSE=717,446.6045 | R²=0.6327 | 本折耗时 0.13s
   折 3/6 | MAE=538,284.4080 | RMSE=757,626.0234 | R²=0.6004 | 本折耗时 0.17s
   折 4/6 | MAE=490,318.2007 | RMSE=696,182.0497 | R²=0.6556 | 本折耗时 0.30s
   折 5/6 | MAE=504,767.4217 | RMSE=688,268.4694 | R²=0.6573 | 本折耗时 0.03s
   折 6/6 | MAE=475,936.4916 | RMSE=655,265.4406 | R²=0.6876 | 本折耗时 0.19s
=== Ridge alpha = 1.0 完成 ===
    -> CV 平均MAE=505,722.1099 ± 20,232.6384
    -> CV 平均MSE=499,640,662,502.8337 ± 44,837,280,637.8329
    -> CV 平均RMSE=706,140.3672 ± 31,724.5074
    -> CV 平均R²=0.6380 ± 0.0329
    -> alpha总耗时 1.01s

=== Ridge alpha = 10.0 开始 ===
   折 1/6 | MAE=520,062.614

In [24]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BEST_ALPHA_RIDGE = 100.0   
KF_SPLITS        = 6
RANDOM_STATE     = 111

print("【Ridge 全量评估】使用 alpha =", BEST_ALPHA_RIDGE, flush=True)

########################################
# 确保 X_ok / y_ok 正常
########################################
if not isinstance(X_ok, pd.DataFrame):
    X_ok = pd.DataFrame(X_ok)

if not isinstance(y_ok, pd.Series):
    y_ok = pd.Series(y_ok)

print("  X_ok.shape =", X_ok.shape, flush=True)
print("  y_ok.shape =", y_ok.shape, flush=True)

########################################
# 1. 切分训练 / 测试
########################################
X_train, X_test, y_train, y_test = train_test_split(
    X_ok,
    y_ok,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("  训练集大小:", X_train.shape, y_train.shape, flush=True)
print("  测试集大小:",  X_test.shape,  y_test.shape,  flush=True)

########################################
# 2. 定义 Ridge 管道
########################################
def make_ridge_pipe(alpha):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("ridge",  Ridge(
            alpha=alpha,
            fit_intercept=True,
            random_state=42
        ))
    ])

ridge_pipe = make_ridge_pipe(BEST_ALPHA_RIDGE)

########################################
# 3. 拟合训练集
########################################
t0 = time.time()
ridge_pipe.fit(X_train, y_train)
print(f"  训练拟合完成，用时 {time.time()-t0:.2f} 秒", flush=True)

########################################
# 4. 样本内 / 样本外 指标
########################################
# in-sample
y_pred_train = ridge_pipe.predict(X_train)

train_MAE  = mean_absolute_error(y_train, y_pred_train)
train_MSE  = mean_squared_error(y_train, y_pred_train)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_train)

# out-of-sample
y_pred_test = ridge_pipe.predict(X_test)

test_MAE  = mean_absolute_error(y_test, y_pred_test)
test_MSE  = mean_squared_error(y_test, y_pred_test)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_test)

print("\n【训练集 / in-sample】", flush=True)
print(f"  MAE  : {train_MAE:,.4f}", flush=True)
print(f"  MSE  : {train_MSE:,.4f}", flush=True)
print(f"  RMSE : {train_RMSE:,.4f}", flush=True)
print(f"  R^2  : {train_R2:.4f}", flush=True)

print("\n【测试集 / out-of-sample】", flush=True)
print(f"  MAE  : {test_MAE:,.4f}", flush=True)
print(f"  MSE  : {test_MSE:,.4f}", flush=True)
print(f"  RMSE : {test_RMSE:,.4f}", flush=True)
print(f"  R^2  : {test_R2:.4f}", flush=True)

########################################
# 5. 6折交叉验证 (全量 X_ok, y_ok)
########################################
print("\n【6折CV / 全量X_ok,y_ok】开始...", flush=True)

kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1
    t_fold = time.time()

    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_ridge_pipe(BEST_ALPHA_RIDGE)
    pipe_cv.fit(X_tr, y_tr)
    y_hat   = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f"    折 {fold_id}/{KF_SPLITS} | "
          f"MAE={mae_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
          f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

cv_MAE_mean  = float(np.mean(cv_mae_list))
cv_MAE_std   = float(np.std(cv_mae_list))
cv_MSE_mean  = float(np.mean(cv_mse_list))
cv_MSE_std   = float(np.std(cv_mse_list))
cv_RMSE_mean = float(np.mean(cv_rmse_list))
cv_RMSE_std  = float(np.std(cv_rmse_list))
cv_R2_mean   = float(np.mean(cv_r2_list))
cv_R2_std    = float(np.std(cv_r2_list))

print("\n【6折CV汇总 / 全量X_ok,y_ok】", flush=True)
print(f"  CV MAE   : {cv_MAE_mean:,.4f}  (std {cv_MAE_std:,.4f})", flush=True)
print(f"  CV MSE   : {cv_MSE_mean:,.4f}  (std {cv_MSE_std:,.4f})", flush=True)
print(f"  CV RMSE  : {cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f})", flush=True)
print(f"  CV R^2   : {cv_R2_mean:.4f}    (std {cv_R2_std:.4f})", flush=True)

########################################
# 6. 系数（Ridge不会稀疏，但可以看方向/大小）
########################################
ridge_model = ridge_pipe.named_steps["ridge"]

coef_table_ridge = pd.DataFrame({
    "feature": X_ok.columns,
    "coef": ridge_model.coef_
}).sort_values(
    by="coef",
    key=lambda s: s.abs(),
    ascending=False
)

print("\n【Ridge 系数Top30(按绝对值排)】", flush=True)
print(coef_table_ridge.head(30).to_string(index=False), flush=True)

coef_table_ridge.to_csv("ridge_coef_full.csv", index=False)
print("  已保存 ridge_coef_full.csv", flush=True)

########################################
# 7. 最后打印一段可直接贴到报告
########################################
print("\n【最后总结 - 直接抄进报告】", flush=True)

print("训练集(in-sample):", flush=True)
print(f" MAE={train_MAE:,.4f} | MSE={train_MSE:,.4f} | RMSE={train_RMSE:,.4f} | R²={train_R2:.4f}", flush=True)

print("测试集(out-of-sample):", flush=True)
print(f" MAE={test_MAE:,.4f} | MSE={test_MSE:,.4f} | RMSE={test_RMSE:,.4f} | R²={test_R2:.4f}", flush=True)

print("6折交叉验证(全量):", flush=True)
print(f" MAE均值={cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f}) | "
      f"MSE均值={cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f}) | "
      f"RMSE均值={cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f}) | "
      f"R²均值={cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

print("\n👍 Ridge 全流程完成。", flush=True)


【Ridge 全量评估】使用 alpha = 100.0
  X_ok.shape = (96048, 66)
  y_ok.shape = (96048,)
  训练集大小: (76838, 66) (76838,)
  测试集大小: (19210, 66) (19210,)
  训练拟合完成，用时 0.18 秒

【训练集 / in-sample】
  MAE  : 485,292.4431
  MSE  : 466,584,007,278.1832
  RMSE : 683,069.5479
  R^2  : 0.6472

【测试集 / out-of-sample】
  MAE  : 484,145.6272
  MSE  : 461,010,298,988.3582
  RMSE : 678,977.3921
  R^2  : 0.6491

【6折CV / 全量X_ok,y_ok】开始...
    折 1/6 | MAE=482,584.9009 | RMSE=674,808.8932 | R²=0.6522 | 用时 0.39s
    折 2/6 | MAE=487,914.5422 | RMSE=687,376.4897 | R²=0.6432 | 用时 0.50s
    折 3/6 | MAE=485,158.7072 | RMSE=683,218.0974 | R²=0.6455 | 用时 0.40s
    折 4/6 | MAE=486,261.3208 | RMSE=685,720.7574 | R²=0.6487 | 用时 0.40s
    折 5/6 | MAE=487,552.3564 | RMSE=685,668.3170 | R²=0.6454 | 用时 0.39s
    折 6/6 | MAE=483,063.3430 | RMSE=679,323.2838 | R²=0.6476 | 用时 0.40s

【6折CV汇总 / 全量X_ok,y_ok】
  CV MAE   : 485,422.5284  (std 2,046.8415)
  CV MSE   : 466,079,095,259.4578  (std 5,930,738,663.0394)
  CV RMSE  : 682,685.9731 (std 4

In [25]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################################
# 可调网格
# alpha 越大，正则越强；l1_ratio 越大，越像Lasso
########################################################
EN_ALPHAS = [1.0, 10.0, 100.0, 1000.0, 10000.0]
EN_L1_RATIOS = [0.2, 0.5, 0.8]   # 0.2更像Ridge，0.8更像Lasso

SUBSET_SIZE = 2000
KF_SPLITS   = 6
SEED        = 42

print("【EN 第0步】开始弹性网络(Elastic Net)小样本调参", flush=True)

########################################################
# 第1步：检查并规整 X_ok / y_ok
########################################################
print("【EN 第1步】检查 X_ok / y_ok", flush=True)

if not isinstance(X_ok, pd.DataFrame):
    print("  X_ok 不是 DataFrame，自动转", flush=True)
    X_ok = pd.DataFrame(X_ok)

if not isinstance(y_ok, pd.Series):
    print("  y_ok 不是 Series，自动转", flush=True)
    y_ok = pd.Series(y_ok)

print("  X_ok.shape =", X_ok.shape, flush=True)
print("  y_ok.shape =", y_ok.shape, flush=True)

########################################################
# 第2步：抽子样本
########################################################
print("【EN 第2步】抽子样本做交叉验证...", flush=True)

n_total = len(X_ok)
n_take  = min(SUBSET_SIZE, n_total)

rng = np.random.RandomState(SEED)
idx = rng.choice(n_total, size=n_take, replace=False)

X_sub = X_ok.iloc[idx].copy()
y_sub = y_ok.iloc[idx].copy()

print(f"  抽到子样本: {len(X_sub)} / {n_total} 行", flush=True)

########################################################
# 第3步：6折划分 + 标准化
########################################################
kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=111)
scaler = StandardScaler()

print("  alpha 候选:", EN_ALPHAS, flush=True)
print("  l1_ratio 候选:", EN_L1_RATIOS, flush=True)

########################################################
# 第4步：双循环 (alpha, l1_ratio) -> 6折CV
########################################################
results_en = []  # 我们把每种组合的平均指标记录下来

print("【EN 第3步】开始网格搜索 (alpha × l1_ratio) ...", flush=True)

for a in EN_ALPHAS:
    for l1r in EN_L1_RATIOS:
        print(f"\n=== ElasticNet alpha={a}, l1_ratio={l1r} 开始 ===", flush=True)
        t_pair = time.time()

        fold_mae_list  = []
        fold_mse_list  = []
        fold_rmse_list = []
        fold_r2_list   = []

        fold_id = 0
        for tr_idx, va_idx in kf.split(X_sub):
            fold_id += 1
            t_fold = time.time()

            X_tr = X_sub.iloc[tr_idx]
            X_va = X_sub.iloc[va_idx]
            y_tr = y_sub.iloc[tr_idx]
            y_va = y_sub.iloc[va_idx]

            pipe = Pipeline([
                ("scaler", scaler),
                ("enet",  ElasticNet(
                    alpha=a,
                    l1_ratio=l1r,
                    fit_intercept=True,
                    max_iter=50000,
                    random_state=42
                ))
            ])

            pipe.fit(X_tr, y_tr)
            y_hat = pipe.predict(X_va)

            mae_val  = mean_absolute_error(y_va, y_hat)
            mse_val  = mean_squared_error(y_va, y_hat)
            rmse_val = np.sqrt(mse_val)
            r2_val   = r2_score(y_va, y_hat)

            fold_mae_list.append(mae_val)
            fold_mse_list.append(mse_val)
            fold_rmse_list.append(rmse_val)
            fold_r2_list.append(r2_val)

            print(f"   折 {fold_id}/{KF_SPLITS} | "
                  f"MAE={mae_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
                  f"| 本折耗时 {time.time()-t_fold:.2f}s", flush=True)

        mae_mean   = float(np.mean(fold_mae_list))
        mae_std    = float(np.std(fold_mae_list))
        mse_mean   = float(np.mean(fold_mse_list))
        mse_std    = float(np.std(fold_mse_list))
        rmse_mean  = float(np.mean(fold_rmse_list))
        rmse_std   = float(np.std(fold_rmse_list))
        r2_mean    = float(np.mean(fold_r2_list))
        r2_std     = float(np.std(fold_r2_list))
        elapsed    = time.time() - t_pair

        print(f"=== ElasticNet alpha={a}, l1_ratio={l1r} 完成 ===", flush=True)
        print(f"    -> CV 平均MAE={mae_mean:,.4f} ± {mae_std:,.4f}", flush=True)
        print(f"    -> CV 平均MSE={mse_mean:,.4f} ± {mse_std:,.4f}", flush=True)
        print(f"    -> CV 平均RMSE={rmse_mean:,.4f} ± {rmse_std:,.4f}", flush=True)
        print(f"    -> CV 平均R²={r2_mean:.4f} ± {r2_std:.4f}", flush=True)
        print(f"    -> 该组合总耗时 {elapsed:.2f}s", flush=True)

        # 记录下来，后面排序
        results_en.append({
            "alpha": a,
            "l1_ratio": l1r,
            "cv_mae_mean": mae_mean,
            "cv_mae_std":  mae_std,
            "cv_mse_mean": mse_mean,
            "cv_mse_std":  mse_std,
            "cv_rmse_mean": rmse_mean,
            "cv_rmse_std":  rmse_std,
            "cv_r2_mean":  r2_mean,
            "cv_r2_std":   r2_std,
            "time_sec": elapsed
        })

########################################################
# 第5步：总结 & 选最优 (按 MAE 均值从小到大)
########################################################
print("\n【EN 第4步】整理结果，选最优组合 (alpha, l1_ratio)", flush=True)

en_results_df = pd.DataFrame(results_en)
en_sorted = en_results_df.sort_values("cv_mae_mean").reset_index(drop=True)

best_row = en_sorted.iloc[0]
best_alpha_en   = best_row["alpha"]
best_l1r_en     = best_row["l1_ratio"]

print("========== 弹性网络小样本CV结果 (按MAE升序) ==========", flush=True)
print(en_sorted.to_string(index=False), flush=True)
print("====================================================", flush=True)

print(f"\n>>> 最优 alpha = {best_alpha_en}, 最优 l1_ratio = {best_l1r_en}", flush=True)

# 你抄下 best_alpha_en 和 best_l1r_en
# 下面第二段全量评估要用它们


【EN 第0步】开始弹性网络(Elastic Net)小样本调参
【EN 第1步】检查 X_ok / y_ok
  X_ok.shape = (96048, 66)
  y_ok.shape = (96048,)
【EN 第2步】抽子样本做交叉验证...
  抽到子样本: 2000 / 96048 行
  alpha 候选: [1.0, 10.0, 100.0, 1000.0, 10000.0]
  l1_ratio 候选: [0.2, 0.5, 0.8]
【EN 第3步】开始网格搜索 (alpha × l1_ratio) ...

=== ElasticNet alpha=1.0, l1_ratio=0.2 开始 ===
   折 1/6 | MAE=596,944.4539 | RMSE=774,222.8952 | R²=0.5335 | 本折耗时 0.11s
   折 2/6 | MAE=574,086.3935 | RMSE=792,492.8297 | R²=0.5519 | 本折耗时 0.30s
   折 3/6 | MAE=608,413.1653 | RMSE=827,467.9393 | R²=0.5234 | 本折耗时 0.30s
   折 4/6 | MAE=590,366.4900 | RMSE=791,672.9882 | R²=0.5547 | 本折耗时 0.20s
   折 5/6 | MAE=582,609.1623 | RMSE=788,255.1212 | R²=0.5504 | 本折耗时 0.30s
   折 6/6 | MAE=570,666.2013 | RMSE=751,595.3648 | R²=0.5890 | 本折耗时 0.30s
=== ElasticNet alpha=1.0, l1_ratio=0.2 完成 ===
    -> CV 平均MAE=587,180.9777 ± 13,047.2616
    -> CV 平均MSE=620,859,502,642.9673 ± 35,949,269,848.1925
    -> CV 平均RMSE=787,617.8564 ± 22,751.1522
    -> CV 平均R²=0.5505 ± 0.0205
    -> 该组合总耗时 1.52s

==

In [16]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

########################################################
# 把这两个值改成你在调参步骤里得到的结果
########################################################
BEST_ALPHA_EN     = 1.0   # ← 用你自己的 best_alpha_en
BEST_L1_RATIO_EN  = 0.8      # ← 用你自己的 best_l1r_en

TEST_SIZE    = 0.2
KF_SPLITS    = 6
RANDOM_STATE = 111

print("【EN 全量评估】alpha =", BEST_ALPHA_EN, " l1_ratio =", BEST_L1_RATIO_EN, flush=True)

########################################################
# 1. 确保 X_ok / y_ok 是 DataFrame / Series
########################################################
if not isinstance(X_ok, pd.DataFrame):
    X_ok = pd.DataFrame(X_ok)

if not isinstance(y_ok, pd.Series):
    y_ok = pd.Series(y_ok)

print("  X_ok.shape =", X_ok.shape, flush=True)
print("  y_ok.shape =", y_ok.shape, flush=True)

########################################################
# 2. 切分训练 / 测试
########################################################
X_train, X_test, y_train, y_test = train_test_split(
    X_ok,
    y_ok,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("  训练集大小:", X_train.shape, y_train.shape, flush=True)
print("  测试集大小:",  X_test.shape,  y_test.shape,  flush=True)

########################################################
# 3. 定义最终弹性网络管道
########################################################
def make_enet_pipe(alpha, l1_ratio):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("enet",  ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            fit_intercept=True,
            max_iter=50000,
            random_state=42
        ))
    ])

enet_pipe = make_enet_pipe(BEST_ALPHA_EN, BEST_L1_RATIO_EN)

########################################################
# 4. 拟合训练集
########################################################
t0 = time.time()
enet_pipe.fit(X_train, y_train)
print(f"  训练拟合完成，用时 {time.time()-t0:.2f} 秒", flush=True)

########################################################
# 5. 训练集 / 测试集 指标
########################################################
# 样本内(训练集)
y_pred_train = enet_pipe.predict(X_train)

train_MAE  = mean_absolute_error(y_train, y_pred_train)
train_MSE  = mean_squared_error(y_train, y_pred_train)
train_RMSE = np.sqrt(train_MSE)
train_R2   = r2_score(y_train, y_pred_train)

# 样本外(测试集)
y_pred_test = enet_pipe.predict(X_test)

test_MAE  = mean_absolute_error(y_test, y_pred_test)
test_MSE  = mean_squared_error(y_test, y_pred_test)
test_RMSE = np.sqrt(test_MSE)
test_R2   = r2_score(y_test, y_pred_test)

print("\n【训练集 / in-sample】", flush=True)
print(f"  MAE  : {train_MAE:,.4f}", flush=True)
print(f"  MSE  : {train_MSE:,.4f}", flush=True)
print(f"  RMSE : {train_RMSE:,.4f}", flush=True)
print(f"  R^2  : {train_R2:.4f}", flush=True)

print("\n【测试集 / out-of-sample】", flush=True)
print(f"  MAE  : {test_MAE:,.4f}", flush=True)
print(f"  MSE  : {test_MSE:,.4f}", flush=True)
print(f"  RMSE : {test_RMSE:,.4f}", flush=True)
print(f"  R^2  : {test_R2:.4f}", flush=True)

########################################################
# 6. 6折交叉验证 (全量 X_ok, y_ok)
########################################################
print("\n【6折CV / 全量X_ok,y_ok】开始...", flush=True)

kf = KFold(n_splits=KF_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_mae_list  = []
cv_mse_list  = []
cv_rmse_list = []
cv_r2_list   = []

fold_id = 0
for tr_idx, va_idx in kf.split(X_ok):
    fold_id += 1
    t_fold = time.time()

    X_tr = X_ok.iloc[tr_idx]
    X_va = X_ok.iloc[va_idx]
    y_tr = y_ok.iloc[tr_idx]
    y_va = y_ok.iloc[va_idx]

    pipe_cv = make_enet_pipe(BEST_ALPHA_EN, BEST_L1_RATIO_EN)
    pipe_cv.fit(X_tr, y_tr)
    y_hat = pipe_cv.predict(X_va)

    mae_val  = mean_absolute_error(y_va, y_hat)
    mse_val  = mean_squared_error(y_va, y_hat)
    rmse_val = np.sqrt(mse_val)
    r2_val   = r2_score(y_va, y_hat)

    cv_mae_list.append(mae_val)
    cv_mse_list.append(mse_val)
    cv_rmse_list.append(rmse_val)
    cv_r2_list.append(r2_val)

    print(f"    折 {fold_id}/{KF_SPLITS} | "
          f"MAE={mae_val:,.4f} | MSE={mse_val:,.4f} | RMSE={rmse_val:,.4f} | R²={r2_val:.4f} "
          f"| 用时 {time.time()-t_fold:.2f}s", flush=True)

########################################################
# 6.5. 计算6折均值和标准差（补完）
########################################################
cv_MAE_mean   = float(np.mean(cv_mae_list))
cv_MAE_std    = float(np.std(cv_mae_list))

cv_MSE_mean   = float(np.mean(cv_mse_list))
cv_MSE_std    = float(np.std(cv_mse_list))

cv_RMSE_mean  = float(np.mean(cv_rmse_list))
cv_RMSE_std   = float(np.std(cv_rmse_list))

cv_R2_mean    = float(np.mean(cv_r2_list))
cv_R2_std     = float(np.std(cv_r2_list))

print("\n【6折CV / 全量X_ok,y_ok】平均结果", flush=True)
print(f"  CV MAE   : {cv_MAE_mean:,.4f}  (std {cv_MAE_std:,.4f})", flush=True)
print(f"  CV MSE   : {cv_MSE_mean:,.4f}  (std {cv_MSE_std:,.4f})", flush=True)
print(f"  CV RMSE  : {cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f})", flush=True)
print(f"  CV R^2   : {cv_R2_mean:.4f}    (std {cv_R2_std:.4f})", flush=True)

########################################################
# 7. 系数（特征重要性）
########################################################
enet_model = enet_pipe.named_steps["enet"]

coef_table_enet = pd.DataFrame({
    "feature": X_ok.columns,
    "coef": enet_model.coef_
}).sort_values(
    by="coef",
    key=lambda s: s.abs(),
    ascending=False
)

print("\n【弹性网络 系数Top30(按绝对值排)】", flush=True)
print(coef_table_enet.head(30).to_string(index=False), flush=True)

coef_table_enet.to_csv("elasticnet_coef_full.csv", index=False)
print("  已保存 elasticnet_coef_full.csv", flush=True)

########################################################
# 8. 最后总结：一口气抄到报告
########################################################
print("\n【最终总结 - 直接抄进报告】", flush=True)

print("训练集(in-sample):", flush=True)
print(f" MAE={train_MAE:,.4f} | MSE={train_MSE:,.4f} | RMSE={train_RMSE:,.4f} | R²={train_R2:.4f}", flush=True)

print("测试集(out-of-sample):", flush=True)
print(f" MAE={test_MAE:,.4f} | MSE={test_MSE:,.4f} | RMSE={test_RMSE:,.4f} | R²={test_R2:.4f}", flush=True)

print("6折交叉验证(全量):", flush=True)
print(f" MAE均值={cv_MAE_mean:,.4f} (std {cv_MAE_std:,.4f}) | "
      f"MSE均值={cv_MSE_mean:,.4f} (std {cv_MSE_std:,.4f}) | "
      f"RMSE均值={cv_RMSE_mean:,.4f} (std {cv_RMSE_std:,.4f}) | "
      f"R²均值={cv_R2_mean:.4f} (std {cv_R2_std:.4f})", flush=True)

print("\n✅ 弹性网络全量评估完成。", flush=True)


【EN 全量评估】alpha = 1.0  l1_ratio = 0.8
  X_ok.shape = (96048, 66)
  y_ok.shape = (96048,)
  训练集大小: (76838, 66) (76838,)
  测试集大小: (19210, 66) (19210,)
  训练拟合完成，用时 3.25 秒

【训练集 / in-sample】
  MAE  : 496,829.7358
  MSE  : 491,544,231,894.7317
  RMSE : 701,102.1551
  R^2  : 0.6283

【测试集 / out-of-sample】
  MAE  : 496,353.2873
  MSE  : 488,077,385,105.9230
  RMSE : 698,625.3539
  R^2  : 0.6285

【6折CV / 全量X_ok,y_ok】开始...
    折 1/6 | MAE=494,746.0401 | MSE=481,775,769,524.7740 | RMSE=694,100.6912 | R²=0.6321 | 用时 3.30s
    折 2/6 | MAE=498,821.8342 | MSE=497,128,827,574.3361 | RMSE=705,073.6327 | R²=0.6246 | 用时 3.80s
    折 3/6 | MAE=495,821.1417 | MSE=490,113,902,112.7277 | RMSE=700,081.3539 | R²=0.6278 | 用时 4.01s
    折 4/6 | MAE=499,527.0184 | MSE=497,335,148,535.0798 | RMSE=705,219.9292 | R²=0.6285 | 用时 3.41s
    折 5/6 | MAE=497,674.1988 | MSE=494,933,046,496.6949 | RMSE=703,514.7806 | R²=0.6267 | 用时 3.50s
    折 6/6 | MAE=494,506.9807 | MSE=486,133,520,068.8949 | RMSE=697,232.7589 | R²=0.6287 |

In [ ]:
# 查看个人持久化工作区文件
!ls /home/mw/project/

In [ ]:
# 查看当前挂载的数据集目录
!ls /home/mw/input/